In [1]:
import torch
import torch.nn as nn

# Modello

In [2]:
model_conv = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0, dilation=1), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(64, 64, kernel_size=3), nn.ReLU(),
    nn.Flatten(),
    nn.LazyLinear(64), nn.ReLU(),   # in_features calcolato al primo forward
    nn.Linear(64, 10)
)

model_conv

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (7): ReLU()
  (8): Flatten(start_dim=1, end_dim=-1)
  (9): LazyLinear(in_features=0, out_features=64, bias=True)
  (10): ReLU()
  (11): Linear(in_features=64, out_features=10, bias=True)
)

In [5]:
# Parametri di un Conv2d = C_in * C_out * k_h * k_w + C_out
#   C_in  = canali in ingresso (3, RGB)
#   C_out = numero di filtri (32)
#   k_h, k_w = dimensioni del kernel (3x3)
#   + C_out = un bias per ogni filtro
# Primo layer: 3 * 32 * 3 * 3 + 32 = 896

# forward con un'immagine fittizia: inizializza LazyLinear e stampa le shape
x = torch.zeros(1, 3, 32, 32)
with torch.no_grad():
    for layer in model_conv:
        x = layer(x)
        n_param = sum(p.numel() for p in layer.parameters())
        print(f'{layer.__class__.__name__:<12} {str(tuple(x.shape[1:])):<16} {n_param:>8,}')


Conv2d       (32, 30, 30)          896
ReLU         (32, 30, 30)            0
MaxPool2d    (32, 15, 15)            0
Conv2d       (64, 13, 13)       18,496
ReLU         (64, 13, 13)            0
MaxPool2d    (64, 6, 6)              0
Conv2d       (64, 4, 4)         36,928
ReLU         (64, 4, 4)              0
Flatten      (1024,)                 0
Linear       (64,)              65,600
ReLU         (64,)                   0
Linear       (10,)                 650


In [6]:
1024*64+64

65600

In [4]:
model_conv

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (7): ReLU()
  (8): Flatten(start_dim=1, end_dim=-1)
  (9): Linear(in_features=1024, out_features=64, bias=True)
  (10): ReLU()
  (11): Linear(in_features=64, out_features=10, bias=True)
)